# 🏦 Régua IF - OPF (Open Finance)
---

## O que é este notebook?
Gera diariamente a base de usuários PicPay elegíveis para comunicação de **Open Finance (OPF)**, identificando em qual Instituição Financeira (IF) cada usuário tem conta além do PicPay.

## Fluxo geral
```
Tabela SS (Self-Service)
       ↓
 [Seção 2] Priorização por banco (df_priorizado)
       ↓
 [Seção 3] Geração diária: NOVOS + LEGADO
       ↓
 Escrita particionada → validation.pp_users_growth_opf
```

## Grupos de saída
| Grupo | Critério |
|-------|----------|
| **NOVO** | Chave `(user_id + bank_name)` apareceu após a última geração |
| **LEGADO** | Chave existente, não comunicada nos últimos `JANELA_LEGADO_DIAS` dias |

## Limite diário
**800k chaves/dia** — NOVOs preenchem primeiro; LEGADO ocupa os slots restantes, priorizados por `bank_name_priority`.

## Links
- [Planilha de IFs](https://docs.google.com/spreadsheets/d/1D2YjMdCapdqYV2-JB6Kzo51FRGI4e5lDz1A04DNi3ro/edit?gid=0#gid=0)
- [Notebook SS (ajuste_ss_users_opf)](https://picpay-principal.cloud.databricks.com/editor/notebooks/665942654981934?o=3048457690269382)


# 1. Configuração
---
> Configurações do Spark, imports e parâmetros. **Altere apenas `1.3 Parâmetros`** para ajustar o comportamento.

## 1.1 Imports

In [0]:
# 1.1 Imports


## 1.2 Configurações Spark
> AQE (Adaptive Query Execution) ativado — o Spark otimiza joins e partições automaticamente em runtime.

In [0]:
# AQE: otimiza plano de execução em runtime (join strategy, partitions)
spark.conf.set('spark.sql.adaptive.enabled', True)
spark.conf.set('spark.sql.shuffle.partitions', 'auto')         # partições ajustadas dinamicamente
spark.conf.set('spark.sql.join.preferSortMergeJoin', True)
spark.conf.set('spark.sql.adaptive.skewedJoin.enabled', True)  # corrige skew automático
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', True)
# Overwrite dinâmico: sobrescreve apenas a partição do dia, não a tabela inteira
spark.conf.set('spark.sql.sources.partitionOverwriteMode', 'dynamic')


## 1.3 Parâmetros
> Todos os parâmetros configuráveis do pipeline estão aqui centralizados.

In [0]:
# Tabela fonte: após inclusão da data mínima de entrada na tabela (1 ano) --> SS
TABELA_FONTE_NAME = "validation.pp_users_growth_opf_communicration_11_03"
CAMPO_DATA_ENTRADA = 'first_transaction'
CAMPO_DATA_ATUALIZACAO_FONTE = 'last_transaction'
# Tabela de histórico: registra das chaves geradas para comunicacao
TABELA_HISTORICO      = 'validation.pp_users_growth_opf'

# Corte da base: Considerando sempre o último ano --> VALIDAR!!
DATA_INICIO_CORTE = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')
# Data de referência para geração (padrão: hoje).
DATA_HOJE = date.today().strftime('%Y-%m-%d')
DATA_FONTE_MINIMA = date.today() - timedelta(days=1)
# Data a ser considerada para as chaves novas (chave = user_is + bank_name)
# se tivermos um delay maior de dias, tbm é corrigido
DATA_NOVA_CHAVE = (date.today() - timedelta(days=2)).strftime("%Y-%m-%d")


# Janela de cooldown do legado: chaves selecionadas nos últimos N dias
# ficam de fora para evitar comunicação repetitiva com o mesmo usuário.
JANELA_LEGADO_DIAS = 10

# Limite diário de chaves na base final (0 = sem limite).
# Novos ocupam os primeiros slots; legado preenche o restante.
LIMITE_DIARIO = 800_000


print(f'Parâmetros carregados:')
print(f'  Fonte:                 {TABELA_FONTE_NAME}')
print(f'  Histórico:             {TABELA_HISTORICO}')
print(f'  Campo atualização:     {CAMPO_DATA_ATUALIZACAO_FONTE}')
print(f'  Data mínima na fonte:  {DATA_FONTE_MINIMA}')
print(f'  Janela legado:         {JANELA_LEGADO_DIAS} dias')
print(f'  Limite diário:         {LIMITE_DIARIO:,} chaves')
print(f'  Data corte:            {DATA_INICIO_CORTE}')
print(f'  Data nova chave:       {DATA_NOVA_CHAVE}')
print(f'  Data geração:          {DATA_HOJE}')

Parâmetros carregados:
  Fonte:           validation.pp_users_growth_opf_communicration_11_03
  Histórico:       validation.pp_users_growth_opf
  Janela legado:   10 dias
  Limite diário:   800,000 chaves
  Data corte:      2025-03-12
  Data nova chave: 2026-03-10
  Data geração:    2026-03-12


OBS.: Se for mudar a data de inicio para 1a entrada fixa, deve-se utilizar a data da ultima transacao como filtros
```
tabela_fonte = (
  spark.table(TABELA_FONTE_NAME)
    .filter(col('CAMPO_DATA_ULTIMA_TRANSACAO') >= DATA_INICIO_CORTE)
)
```

In [0]:
tabela_fonte = (
  spark.table(TABELA_FONTE_NAME)
    .filter(col(CAMPO_DATA_ENTRADA) >= DATA_INICIO_CORTE)
)

row_fonte_atualizacao = tabela_fonte.selectExpr(
    f"max(date({CAMPO_DATA_ATUALIZACAO_FONTE})) as dt_max_fonte"
).first()
dt_max_fonte = row_fonte_atualizacao['dt_max_fonte']

if dt_max_fonte is None or dt_max_fonte < DATA_FONTE_MINIMA:
    raise RuntimeError(
        f"Fonte {TABELA_FONTE_NAME} desatualizada: max({CAMPO_DATA_ATUALIZACAO_FONTE})={dt_max_fonte}. "
        f"Esperado pelo menos {DATA_FONTE_MINIMA}. Nenhuma partição será gravada em {TABELA_HISTORICO}."
    )

print(
    f"Fonte validada com sucesso: max({CAMPO_DATA_ATUALIZACAO_FONTE})={dt_max_fonte} "
    f"(mínimo esperado: {DATA_FONTE_MINIMA})."
)

tabela_fonte.createOrReplaceTempView('TABELA_FONTE')

# 2. Priorização por Banco
---
> Classifica cada usuário elegível (MAU) pelo banco mais prioritário. O campo `bank_name_priority` define a ordem (1 = maior prioridade). O campo `n_priority = 1` marca a melhor chave por usuário.

In [0]:

# --- VERIFICAR! --> Como vem nas tabelas a serem usadas para incluir no CASE!
# CHECKED = Mesma nomenclatura na SS

bank_flag = 'bank_name'

classif_bank_priority = f"""

CASE
    WHEN {bank_flag} = 'MERCADO PAGO'        THEN 1  -- CHECKED!
    WHEN {bank_flag} = 'NUBANK'              THEN 2  -- CHECKED!
    WHEN {bank_flag} = 'RECARGAPAY'          THEN 3  -- CHECKED!
    WHEN {bank_flag} = 'SHOPEE'              THEN 4  -- CHECKED!
    WHEN {bank_flag} = 'SANTANDER'           THEN 5  -- CHECKED!
    WHEN {bank_flag} = 'CAIXA'               THEN 6  -- CHECKED!
    WHEN {bank_flag} = 'INTER'               THEN 7  -- CHECKED!
    WHEN {bank_flag} = 'BRADESCO'            THEN 8  -- CHECKED!
    WHEN {bank_flag} = 'BANCO DO BRASIL'     THEN 9  -- CHECKED!
    WHEN {bank_flag} = 'PAN'                 THEN 10 -- CHECKED!
    WHEN {bank_flag} = 'NEXT'                THEN 11 -- CHECKED!
    WHEN {bank_flag} = 'NEON'                THEN 12 -- CHECKED!
    WHEN {bank_flag} = 'C6 BANK'             THEN 13 -- CHECKED!
    WHEN {bank_flag} = '99PAY'               THEN 14 -- CHECKED!
    WHEN {bank_flag} = 'BANCO DO NORDESTE'   THEN 15 -- CHECKED!
    WHEN {bank_flag} = 'MERCANTIL'           THEN 16 -- CHECKED!
    WHEN {bank_flag} = 'BRB'                 THEN 17 -- CHECKED!
    WHEN {bank_flag} = 'CREFISA'             THEN 18 -- CHECKED!
    WHEN {bank_flag} = 'MIDWAY'              THEN 19 -- CHECKED!
    WHEN {bank_flag} = 'PAGSEGURO'           THEN 20 -- CHECKED!
    WHEN {bank_flag} = 'WILL BANK'           THEN 21 -- CHECKED!
    WHEN {bank_flag} = 'SAFRA'               THEN 22 -- CHECKED!
    WHEN {bank_flag} = 'BTG'                 THEN 23 -- CHECKED!
    WHEN {bank_flag} = 'BANCO CARREFOUR'     THEN 24 -- CHECKED!
ELSE 100 END
"""

lista_bancos_prioritarios = (
    'MERCADO PAGO',
    'NUBANK',
    'RECARGAPAY',
    'SHOPEE',
    'SANTANDER',
    'CAIXA',
    'INTER',
    'BRADESCO',
    'BANCO DO BRASIL',
    'PAN',
    'NEXT',
    'NEON',
    'C6 BANK',
    '99PAY',
    'BANCO DO NORDESTE',
    'MERCANTIL',
    'BRB',
    'CREFISA',
    'MIDWAY',
    'PAGSEGURO',
    'WILL BANK',
    'SAFRA',
    'BTG',
    'BANCO CARREFOUR'
)

In [0]:
# Tabela fonte: gerada pelo df_priorizado (cell 11) — contém todos os
# usuários elegíveis com prioridade de banco já calculada.
df_priorizado = spark.sql(f"""
with 
base as (
  select 
    user_id,
    case when bank_name not in {lista_bancos_prioritarios} then 'OUTRO' else bank_name end as bank_name,
    date(first_transaction) as dt_entry, -- OBS.: Se for mudar a data de inicio para 1a entrada fixa, deve-se utilizar a data da ultima transacao como filtros
    date(last_transaction) as dt_last_transaction,
    case 
      when first_transaction >= '{DATA_NOVA_CHAVE}' then 1
      else 0 
    end as new_key,
    {classif_bank_priority} as bank_name_priority  -- calculado uma única vez
  from TABELA_FONTE as a
  inner join consumers.dim_consumers_metrics as b
    on a.user_id = b.consumer_id
      and b.is_mau = true
  where first_transaction is not null
)

select 
    user_id,
    bank_name,
    dt_entry,
    dt_last_transaction,
    new_key,
    bank_name_priority,
    row_number() over(
        partition by user_id
        order by case when new_key = 1 then 0 else 1 end, bank_name_priority
        ) as n_priority
    -- dense_rank() over(
    --     partition by user_id 
    --     order by case when new_key = 1 then 1 else bank_name_priority+1 end
    -- ) as n_priority_option_2
from base

""")

df_priorizado.createOrReplaceTempView('df_priorizado')

# Validação - user_id: 248766678519382969,546680897119988940,241693100016315216,595181959380165740,620688672208297813


#### Validação de Priorização

In [0]:
# %sql
# select a.* 
# from df_priorizado as a
# where not exists (
#     select b.*
#     from df_priorizado as b
#     where a.user_id = b.user_id 
#         and new_key = 0
# )
# and a.new_key = 1 

#### Validação de Novos x Estoque


In [0]:
# %skip
# spark.sql(f"""

# with base_novos as (
#   select 
#     '{DATA_NOVA_CHAVE}' as Data,
#     'Novo' as Tipo,
#     case when bank_name in {lista_bancos_prioritarios} then 'PRIORIDADE' else 'OUTRO' end as Prioridade,
#     bank_name as `Instituição Financeira`,
#     count(*) as Chaves,
#     round(100 * count(*) / sum(count(*))over() , 2) as `% Total`
#   from df_priorizado
#   where new_key = 1
#   group by 1,2,3,4
#   order by 1 desc, {classif_bank_priority}       
#   ),
  
#   base_estoque as (
#   select 
#     dt_last_transaction as Data,
#     'Estoque' as Tipo,
#     case when bank_name in {lista_bancos_prioritarios} then 'PRIORIDADE' else 'OUTRO' end as Prioridade,
#     -- n_priority as Prioridade,
#     bank_name as Banco,
#     count(*) as Chaves,
#     round(100 * count(*) / sum(count(*))over() , 2) as `% Total`
#   from df_priorizado
#   where new_key = 0
#   group by 1,2,3,4
#   order by 1 desc, {classif_bank_priority}
#   )

#   select
#     Tipo,
#     Prioridade,
#     sum(Chaves) as Chaves,
#     round(100 * sum(Chaves) / sum(sum(Chaves))over() , 2) as `% Total`
#   from (
#     select * from base_novos
#     union all
#     select * from base_estoque
#   )
#   group by 1,2

# """).display()


# 3. Geração Diária da Base OPF
---

Lógica de seleção diária composta por dois grupos:

- **NOVOS**: todas as chaves `(user_id + bank_name)` que apareceram na base desde a última geração.
- **LEGADO**: chaves já existentes que **não foram selecionadas** nos últimos `JANELA_LEGADO_DIAS` dias, priorizadas por `bank_name_priority` (menor número = maior prioridade).

> Os parâmetros configuráveis estão na célula abaixo.

In [0]:
# Tenta buscar a última data no histórico. Se a tabela não existir ainda
# (primeira execução), usa antes de ontem como fallback.
try:
    dt_ultima_geracao = spark.sql(f"""
        SELECT COALESCE(MAX(dt_geracao), date_sub(current_date(), 2)) AS dt_ultima_geracao
        FROM {TABELA_HISTORICO}
    """).first()['dt_ultima_geracao']
except Exception:
    dt_ultima_geracao = (date.today() - timedelta(days=2)).strftime('%Y-%m-%d')

print(f'Última geração encontrada: {dt_ultima_geracao}')
print(f'Chaves novas = dt_entry > {dt_ultima_geracao}')

Última geração encontrada: 2026-03-10
Chaves novas = dt_entry > 2026-03-10


In [0]:
# Novas = apareceram na base (dt_entry) APÓS a última geração.
# Todas as chaves novas entram na base do dia, sem limite.
df_novos_dia = spark.sql(f"""
    select
        user_id,
        bank_name,
        -- bank_name_priority
        dt_entry,
        dt_last_transaction,
        'NOVO'          as tipo
    from df_priorizado
    where 1=1
        and n_priority = 1             -- melhor banco por usuário
        -- OBS.: Se for mudar a data de inicio para 1a entrada fixa, deve-se utilizar a data da ultima transacao como filtros
        and dt_entry >= '{dt_ultima_geracao}'   -- apareceu após a última geração 
""")

df_novos_dia.createOrReplaceTempView('novos_dia')

df_novos_dia.cache()  # evita recomputo do DAG ao usar como TempView na cell seguinte
qtd_novos = df_novos_dia.count()
print(f'Chaves NOVAS encontradas: {qtd_novos:,}')
print(f'Chaves NOVAS: a partir de {dt_ultima_geracao}')

Chaves NOVAS encontradas: 27,181
Chaves NOVAS: a partir de 2026-03-10


In [0]:
# users (não, chaves) selecionados nos últimos JANELA_LEGADO_DIAS ficam de fora do legado.
try:
    df_historico_recente = spark.sql(f"""
        SELECT DISTINCT user_id --, bank_name
        FROM {TABELA_HISTORICO}
        WHERE dt_geracao >= date_sub('{DATA_HOJE}', {JANELA_LEGADO_DIAS})
    """)
    df_historico_recente.createOrReplaceTempView('historico_recente')
    qtd_historico = df_historico_recente.count()
    print(f'Chaves em cooldown (últimos {JANELA_LEGADO_DIAS} dias): {qtd_historico:,}')
except Exception:
    # Primeira execução: histórico vazio, nenhum legado em cooldown
    spark.sql("CREATE OR REPLACE TEMP VIEW historico_recente AS SELECT '' AS user_id, '' AS bank_name WHERE 1=0")
    print('Histórico não encontrado — nenhum legado em cooldown (primeira execução).')


Histórico não encontrado — nenhum legado em cooldown (primeira execução).


In [0]:
# Legado: usuários elegíveis fora do cooldown, sem duplicar com os novos do dia.
# O ORDER BY é feito na célula seguinte via .orderBy().limit() (TakeOrdered — mais eficiente).
# Legado = chaves que NÃO são novas + NÃO estão em cooldown.
# Ordenadas por bank_name_priority (menor número = banco mais relevante)
# para garantir que o limite diário favorece os bancos prioritários.

df_legado_dia = spark.sql(f"""
    SELECT
        f.user_id,
        f.bank_name,
        f.bank_name_priority,  -- necessário para .orderBy() na cell seguinte
        f.dt_entry,
        f.dt_last_transaction,
        'LEGADO'        AS tipo
    FROM df_priorizado AS f
    -- Exclui users (não, chaves) que já estão em cooldown (selecionadas nos últimos N dias)
    LEFT JOIN historico_recente AS h
        ON  f.user_id   = h.user_id
        -- AND f.bank_name = h.bank_name
    -- Exclui users (não, chaves) que já entraram como NOVOS no dia de hoje
    LEFT JOIN novos_dia AS n
        ON  f.user_id   = n.user_id
        -- AND f.bank_name = n.bank_name
    WHERE f.n_priority = 1             -- melhor banco por usuário
      AND f.dt_entry <= '{dt_ultima_geracao}'  -- não é novo (existia antes da última geração)
      AND h.user_id IS NULL                    -- não está em cooldown
      AND n.user_id IS NULL                    -- não duplica com os novos do dia
""")

In [0]:
# Novos sempre entram todos. Legado preenche os slots restantes até o limite.
if LIMITE_DIARIO > 0:
    slots_legado = max(0, LIMITE_DIARIO - qtd_novos)
    df_legado_limitado = (
        df_legado_dia
        # bancos mais prioritários primeiro (TEM QUE VER SE QUEREM ALEATORIZAR OU PEGAR PROPORÇÃO ENTRE BANCOS)
        .orderBy('bank_name_priority', ascending=True)  # bancos prioritários primeiro
        .limit(slots_legado)
    )
    print(f'Slots disponíveis para legado: {slots_legado:,} (limite {LIMITE_DIARIO:,} - {qtd_novos:,} novos)')
else:
    df_legado_limitado = df_legado_dia
    print('Sem limite diário — todo o legado elegível será incluído.')

df_base_diaria = df_novos_dia.unionAll(df_legado_limitado)

Slots disponíveis para legado: 772,819 (limite 800,000 - 27,181 novos)


In [0]:
_output_df_ = df_base_diaria\
    .select(
        (monotonically_increasing_id() + 1).alias('pp_users_growth_opf'),
        'user_id',
        'bank_name',
        'dt_entry',
        'dt_last_transaction',
        col('tipo').alias('type'),
        to_date(lit(DATA_HOJE)).alias('updated_date')
    )

# Cache antes de usar em resumo + write (evita recomputar o DAG 3 vezes)
_output_df_.cache()


In [0]:
from pyspark.sql.functions import expr

resumo = (
    _output_df_
        .groupBy('type')
        .agg(
            count('*').alias('chaves'),
            countDistinct('user_id').alias('usuarios'),
            expr("round(100.0 * count(*) / sum(count(*)) over (), 2)").alias('pct_total')
        )
)


print(f'=== Geração {DATA_HOJE} ===')
print(f'Última geração anterior: {dt_ultima_geracao}')
resumo.display()

=== Geração 2026-03-12 ===
Última geração anterior: 2026-03-10


In [0]:
_output_df_.write\
    .mode('overwrite')\
    .partitionBy('updated_date')\
    .saveAsTable(TABELA_HISTORICO)

In [ ]:
# Libera memória após gravação
_output_df_.unpersist()
df_novos_dia.unpersist()
print('Cache liberado.')
